# PIK Structural Credit Pricing

**Question:** What are the breakeven spreads between cash-pay, full-PIK, and PIK-toggle bonds across issuers of varying credit riskiness?

This notebook demonstrates the full structural credit + Monte Carlo pipeline:

1. **Merton Model** — Distance-to-default (DD), default probability (PD), and implied spreads from balance sheet data
2. **Endogenous Hazard** — Hazard rates that increase as PIK accrual raises leverage
3. **Dynamic Recovery** — Recovery rates that decline as notional grows
4. **Toggle Exercise** — Threshold-based and optimal (nested MC) PIK/cash decisions
5. **MC Pricing** — Full path simulation with first-passage default and PIK toggle logic

In [1]:
from __future__ import annotations
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.valuations.instruments import (
    Bond,
    MertonModel,
    MertonAssetDynamics,
    MertonBarrierType,
    EndogenousHazardSpec,
    DynamicRecoverySpec,
    ToggleExerciseModel,
    MertonMcConfig,
    MertonMcResult,
)

## 1. Setup: Bond and Market Parameters

In [2]:
# Market parameters
RISK_FREE = 0.045       # 4.5% base rate
COUPON    = 0.085       # 8.5% annual coupon (typical HY PIK)
MATURITY  = 5           # years
NOTIONAL  = 100.0
AS_OF     = date(2025, 6, 15)
MAT_DATE  = AS_OF + timedelta(days=int(MATURITY * 365.25))
NUM_PATHS = 25_000
SEED      = 42

# Build the bond once — same structure for all issuers
bond = (
    Bond.builder("PIK-DEMO")
    .money(Money(NOTIONAL, USD))
    .coupon_rate(COUPON)
    .issue(AS_OF)
    .maturity(MAT_DATE)
    .frequency(2)          # semi-annual
    .disc_id("USD-OIS")
    .build()
)

print(f"Bond: {MATURITY}Y  {COUPON:.1%} semi-annual  |  Notional: {NOTIONAL}  |  Maturity: {MAT_DATE}")

Bond: 5Y  8.5% semi-annual  |  Notional: 100.0  |  Maturity: 2030-06-15


## 2. Issuer Profiles

We define five issuers spanning the credit spectrum.  Each is characterised by:
- **Asset value** — market value of the firm's assets (debt barrier = 100, so asset / 100 = coverage)
- **Asset vol** — annualised volatility of asset returns (higher vol = riskier)
- **Base hazard** — calibrated starting hazard rate for endogenous model
- **Base recovery** — expected recovery at par notional

In [3]:
PROFILES = [
    {"name": "BB+ (Solid HY)",       "V": 200, "vol": 0.20, "B": 100, "h0": 0.015, "R0": 0.45},
    {"name": "BB- (Mid HY)",         "V": 165, "vol": 0.25, "B": 100, "h0": 0.035, "R0": 0.40},
    {"name": "B  (Weak HY)",         "V": 140, "vol": 0.30, "B": 100, "h0": 0.060, "R0": 0.35},
    {"name": "B- (Stressed)",        "V": 125, "vol": 0.35, "B": 100, "h0": 0.090, "R0": 0.30},
    {"name": "CCC (Deeply Stressed)","V": 115, "vol": 0.40, "B": 100, "h0": 0.140, "R0": 0.25},
]

print(f"{'Issuer':<25s}  {'Assets':>6s}  {'Vol':>5s}  {'Cover':>5s}  {'h0':>5s}  {'R0':>5s}")
print("-" * 65)
for p in PROFILES:
    print(f"{p['name']:<25s}  {p['V']:>6.0f}  {p['vol']:>5.0%}  {p['V']/p['B']:>5.2f}x {p['h0']:>5.1%}  {p['R0']:>5.0%}")

Issuer                     Assets    Vol  Cover     h0     R0
-----------------------------------------------------------------
BB+ (Solid HY)                200    20%   2.00x  1.5%    45%
BB- (Mid HY)                  165    25%   1.65x  3.5%    40%
B  (Weak HY)                  140    30%   1.40x  6.0%    35%
B- (Stressed)                 125    35%   1.25x  9.0%    30%
CCC (Deeply Stressed)         115    40%   1.15x 14.0%    25%


## 3. Merton Model: Analytical Credit Metrics

Before running Monte Carlo, the Merton model gives us closed-form distance-to-default and default probability.  The model treats equity as a call option on firm assets with strike = debt barrier.

In [4]:
print(f"{'Issuer':<25s}  {'DD':>6s}  {'PD(5Y)':>7s}  {'Impl Sprd':>9s}")
print("-" * 55)

for p in PROFILES:
    m = MertonModel(
        asset_value=p["V"],
        asset_vol=p["vol"],
        debt_barrier=p["B"],
        risk_free_rate=RISK_FREE,
        barrier_type=MertonBarrierType.first_passage(0.0),
    )
    dd = m.distance_to_default(MATURITY)
    pd = m.default_probability(MATURITY)
    spread = m.implied_spread(MATURITY, p["R0"])

    print(f"{p['name']:<25s}  {dd:>6.2f}  {pd:>7.2%}  {spread:>7.0f} bp")

Issuer                         DD   PD(5Y)  Impl Sprd
-------------------------------------------------------
BB+ (Solid HY)               1.83   41.12%        0 bp
BB- (Mid HY)                 1.02   78.01%        0 bp
B  (Weak HY)                 0.50  100.00%        0 bp
B- (Stressed)                0.18  111.91%        0 bp
CCC (Deeply Stressed)       -0.04  119.35%        0 bp


## 4. Endogenous Hazard and Dynamic Recovery

PIK bonds create a **feedback loop**: PIK accrual grows the notional, which increases leverage, which raises the hazard rate and lowers recovery.  These two modules capture this:

- **Endogenous Hazard** (power law): $\lambda(L) = \lambda_0 \cdot (L / L_0)^\beta$
- **Dynamic Recovery** (floored inverse): $R(N) = \max(\text{floor}, R_0 \cdot N_0 / N)$

In [5]:
# Demonstrate the hazard feedback for the B- issuer
p = PROFILES[3]  # B- (Stressed)
endo = EndogenousHazardSpec.power_law(
    base_hazard=p["h0"],
    base_leverage=p["B"] / p["V"],  # 0.80
    exponent=2.0,
)

dyn_rec = DynamicRecoverySpec.floored_inverse(
    base_recovery=p["R0"],
    base_notional=NOTIONAL,
    floor=0.10,
)

print(f"B- Issuer:  base leverage = {p['B']/p['V']:.2f},  base hazard = {p['h0']:.1%}")
print()
print(f"{'Leverage':>10s}  {'Hazard':>8s}  |  {'Notional':>10s}  {'Recovery':>8s}")
print("-" * 48)
for lev in [0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20]:
    h = endo.hazard_at_leverage(lev)
    ntl = NOTIONAL * (lev / (p["B"] / p["V"]))  # implied notional at this leverage
    r = dyn_rec.recovery_at_notional(ntl)
    print(f"{lev:>10.2f}  {h:>8.2%}  |  {ntl:>10.1f}  {r:>8.2%}")

B- Issuer:  base leverage = 0.80,  base hazard = 9.0%

  Leverage    Hazard  |    Notional  Recovery
------------------------------------------------
      0.60     5.06%  |        75.0    30.00%
      0.70     6.89%  |        87.5    30.00%
      0.80     9.00%  |       100.0    30.00%
      0.90    11.39%  |       112.5    26.67%
      1.00    14.06%  |       125.0    24.00%
      1.10    17.02%  |       137.5    21.82%
      1.20    20.25%  |       150.0    20.00%


## 5. Monte Carlo Pricing: Cash vs Full PIK vs Toggle

We now run the full MC engine for each issuer and each bond structure:

1. **Cash-Pay** — No PIK features, plain structural credit MC
2. **Full PIK** — All coupons accrete to notional; endogenous hazard + dynamic recovery enabled
3. **PIK Toggle** — Borrower elects PIK when hazard rate exceeds 10% threshold

In [6]:
def make_merton(p: dict) -> MertonModel:
    return MertonModel(
        asset_value=p["V"],
        asset_vol=p["vol"],
        debt_barrier=p["B"],
        risk_free_rate=RISK_FREE,
        barrier_type=MertonBarrierType.first_passage(0.0),
    )


def make_configs(p: dict) -> tuple[MertonMcConfig, MertonMcConfig, MertonMcConfig]:
    m = make_merton(p)
    endo = EndogenousHazardSpec.power_law(p["h0"], p["B"] / p["V"], 2.0)
    drec = DynamicRecoverySpec.floored_inverse(p["R0"], NOTIONAL, 0.10)
    toggle = ToggleExerciseModel.threshold("hazard_rate", 0.10, "above")

    kw = dict(num_paths=NUM_PATHS, seed=SEED, antithetic=True, time_steps_per_year=12)

    cash   = MertonMcConfig(merton=m, **kw)
    pik    = MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec, **kw)
    toggle_cfg = MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec, toggle_model=toggle, **kw)

    return cash, pik, toggle_cfg

In [7]:
# Run the MC engine for all profiles
results = {}  # {issuer_name: {structure: MertonMcResult}}

for p in PROFILES:
    cash_cfg, pik_cfg, toggle_cfg = make_configs(p)
    results[p["name"]] = {
        "Cash-Pay":   bond.price_merton_mc(config=cash_cfg,   discount_rate=RISK_FREE, as_of=AS_OF),
        "Full PIK":   bond.price_merton_mc(config=pik_cfg,    discount_rate=RISK_FREE, as_of=AS_OF),
        "PIK Toggle": bond.price_merton_mc(config=toggle_cfg, discount_rate=RISK_FREE, as_of=AS_OF),
    }

print(f"Priced {len(PROFILES)} issuers x 3 structures = {len(PROFILES) * 3} MC runs")

Priced 5 issuers x 3 structures = 15 MC runs

## 6. Results: Detailed View

In [8]:
for issuer, structs in results.items():
    print(f"\n{'=' * 85}")
    print(f"  {issuer}")
    print(f"{'=' * 85}")
    print(f"  {'Structure':<15s}  {'Price':>7s}  {'Spread':>7s}  {'E[Loss]':>7s}  "
          f"{'DefRate':>7s}  {'PIK%':>6s}  {'TermNtl':>8s}  {'SE':>6s}")
    print(f"  {'-' * 75}")
    for name, r in structs.items():
        print(f"  {name:<15s}  {r.clean_price_pct:7.2f}  {r.effective_spread_bp:6.0f}bp  "
              f"{r.expected_loss:7.2%}  {r.default_rate:7.2%}  {r.average_pik_fraction:5.1%}  "
              f"{r.avg_terminal_notional:8.1f}  {r.standard_error:6.4f}")


  BB+ (Solid HY)
  Structure          Price   Spread  E[Loss]  DefRate    PIK%   TermNtl      SE
  ---------------------------------------------------------------------------
  Cash-Pay          113.53    -704bp    3.37%    6.74%   0.0%     100.0  0.0009
  Full PIK          113.81    -709bp    3.12%    6.74%   0.0%     100.0  0.0009
  PIK Toggle        113.81    -709bp    3.12%    6.74%   0.0%     100.0  0.0009

  BB- (Mid HY)
  Structure          Price   Spread  E[Loss]  DefRate    PIK%   TermNtl      SE
  ---------------------------------------------------------------------------
  Cash-Pay           99.02    -430bp   15.71%   28.99%   0.0%     100.0  0.0018
  Full PIK           99.02    -430bp   15.71%   28.99%   0.0%     100.0  0.0018
  PIK Toggle         99.02    -430bp   15.71%   28.99%   0.0%     100.0  0.0018

  B  (Weak HY)
  Structure          Price   Spread  E[Loss]  DefRate    PIK%   TermNtl      SE
  ------------------------------------------------------------------------

## 7. Breakeven Spread Summary

The key question: **How much extra spread should an investor demand for PIK risk?**

In [9]:
print(f"{'Issuer':<25s}  {'Cash bp':>8s}  {'PIK bp':>8s}  {'Toggle bp':>9s}  "
      f"{'PIK-Cash':>8s}  {'Tog-Cash':>8s}")
print("-" * 80)

for issuer, structs in results.items():
    c = structs["Cash-Pay"].effective_spread_bp
    p = structs["Full PIK"].effective_spread_bp
    t = structs["PIK Toggle"].effective_spread_bp
    print(f"{issuer:<25s}  {c:>8.0f}  {p:>8.0f}  {t:>9.0f}  {p - c:>+8.0f}  {t - c:>+8.0f}")

print()
print("PIK-Cash  = additional spread for full-PIK vs cash-pay (the pure PIK premium)")
print("Tog-Cash  = additional spread for toggle vs cash-pay")

Issuer                      Cash bp    PIK bp  Toggle bp  PIK-Cash  Tog-Cash
--------------------------------------------------------------------------------
BB+ (Solid HY)                 -704      -709       -709        -5        -5
BB- (Mid HY)                   -430      -430       -430        +0        +0
B  (Weak HY)                     22        89        106       +67       +84
B- (Stressed)                   453       689        743      +236      +290
CCC (Deeply Stressed)           773      1288       1388      +515      +615

PIK-Cash  = additional spread for full-PIK vs cash-pay (the pure PIK premium)
Tog-Cash  = additional spread for toggle vs cash-pay


## 7b. Hazard-Rate Model: PIK Pricing Without Structural Feedback

Before the full Merton simulation, we can ask a simpler question: **how much of the PIK premium comes purely from the timing and notional effects?**

A **flat hazard rate** model uses constant $\lambda$ and the library's `HazardCurve` and `DiscountCurve` term structures:

- `HazardCurve.survival(t)` $= S(t) = e^{-\lambda t}$ — probability of no default by time $t$
- `HazardCurve.default_prob(t_1, t_2)` $= S(t_1) - S(t_2)$ — default probability in each period
- `DiscountCurve.df(t)` $= D(t) = e^{-r t}$ — risk-free discount factor

**Cash-pay:** Coupons arrive at $t_1, \ldots, t_n$ — each weighted by $S(t_i) \cdot D(t_i)$, plus principal at maturity.
**Full PIK:** No coupon cash flows. At maturity, receive $N \cdot (1 + c/f)^n$ — the entire PIK-inflated notional, weighted by $S(T) \cdot D(T)$.

The PIK bond concentrates all value at the longest maturity (lowest survival probability) with a larger notional (higher loss given default). This captures the **timing + notional** effect but **not** the endogenous feedback loop (PIK → higher leverage → higher $\lambda$).

We show two views:
1. **Price comparison** — at each issuer's base $\lambda$, how do HR prices compare to MC prices?
2. **Implied hazard rates** — given the MC model's prices, what flat $\lambda$ reproduces them? The PIK implied $\lambda$ exceeds the cash implied $\lambda$ by the **structural hazard premium**.

In [10]:
import math
from finstack.core.market_data.term_structures import DiscountCurve, HazardCurve

# -- Build shared discount curve (flat risk-free rate) --------------------
dc = DiscountCurve(
    "USD-OIS", AS_OF,
    [(t, math.exp(-RISK_FREE * t))
     for t in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]],
)

# -- Hazard-rate pricing using library HazardCurve + DiscountCurve --------

def hr_price_cash(hc: HazardCurve, dc: DiscountCurve) -> float:
    """PV of a cash-pay bond using library curve objects."""
    n = MATURITY * 2  # semi-annual
    dt = 0.5
    cpn = COUPON / 2 * NOTIONAL
    pv = 0.0
    for i in range(1, n + 1):
        t = i * dt
        surv = hc.survival(t)
        df = dc.df(t)
        cf = cpn + (NOTIONAL if i == n else 0.0)
        pv += cf * surv * df
        pv += hc.recovery_rate * NOTIONAL * hc.default_prob(t - dt, t) * df
    return pv

def hr_price_pik(hc: HazardCurve, dc: DiscountCurve) -> float:
    """PV of a full-PIK bond (growing notional, no coupon cash flows)."""
    n = MATURITY * 2
    dt = 0.5
    g = 1.0 + COUPON / 2
    pv = 0.0
    for i in range(1, n + 1):
        t = i * dt
        surv = hc.survival(t)
        df = dc.df(t)
        ntl = NOTIONAL * g ** i
        if i == n:
            pv += ntl * surv * df
        pv += hc.recovery_rate * ntl * hc.default_prob(t - dt, t) * df
    return pv

def find_implied_hazard(price_fn, target, recovery):
    """Bisection: find flat lambda that prices to target PV."""
    lo, hi = 0.0, 5.0
    for _ in range(200):
        mid = (lo + hi) / 2
        hc = HazardCurve("CREDIT", AS_OF, [(0.0, mid), (10.0, mid)],
                         recovery_rate=recovery)
        if price_fn(hc, dc) > target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

# -- View 1: prices at each issuer's base hazard rate --------------------

print("HAZARD-RATE PRICES: Cash-Pay vs PIK at Each Issuer's Flat lambda")
print("=" * 100)
print(f"  {'Issuer':<25s}  {'lam(bp)':>7s}  "
      f"{'HR Cash':>9s}  {'HR PIK':>9s}  {'d Price':>8s}  "
      f"{'MC Cash':>9s}  {'MC PIK':>9s}  {'d Price':>8s}")
print("  " + "-" * 90)

for p in PROFILES:
    lam, rec = p["h0"], p["R0"]
    hc = HazardCurve("CREDIT", AS_OF, [(0.0, lam), (10.0, lam)],
                     recovery_rate=rec)
    hr_cash = hr_price_cash(hc, dc)
    hr_pik  = hr_price_pik(hc, dc)

    mc_c = results[p["name"]]["Cash-Pay"].clean_price_pct
    mc_p = results[p["name"]]["Full PIK"].clean_price_pct

    print(f"  {p['name']:<25s}  {lam*1e4:7.0f}  "
          f"{hr_cash:9.2f}  {hr_pik:9.2f}  {hr_pik - hr_cash:+8.2f}  "
          f"{mc_c:9.2f}  {mc_p:9.2f}  {mc_p - mc_c:+8.2f}")

print()
print("  d Price = PIK minus Cash (negative = PIK trades cheaper)")
print("  HR captures timing + notional; MC adds endogenous feedback.")

# -- View 2: implied hazard rates from MC prices -------------------------

print()
print("IMPLIED HAZARD RATES: Flat lambda Backing Out Each Merton MC Price")
print("=" * 100)
print(f"  {'Issuer':<25s}  {'Base lam':>8s}  "
      f"{'lam Cash':>8s}  {'lam PIK':>8s}  {'d lam':>7s}  "
      f"{'MC Cash':>8s}  {'MC PIK':>8s}  {'d Sprd':>7s}")
print("  " + "-" * 85)

for p in PROFILES:
    rec = p["R0"]
    mc_cash = results[p["name"]]["Cash-Pay"]
    mc_pik  = results[p["name"]]["Full PIK"]

    # Target PV from MC prices (price % → currency units)
    cash_target = mc_cash.clean_price_pct / 100 * NOTIONAL
    pik_target  = mc_pik.clean_price_pct / 100 * NOTIONAL

    lam_c = find_implied_hazard(hr_price_cash, cash_target, rec)
    lam_p = find_implied_hazard(hr_price_pik, pik_target, rec)

    print(f"  {p['name']:<25s}  {p['h0']*1e4:8.0f}  "
          f"{lam_c*1e4:8.0f}  {lam_p*1e4:8.0f}  {(lam_p - lam_c)*1e4:+7.0f}  "
          f"{mc_cash.effective_spread_bp:7.0f}bp {mc_pik.effective_spread_bp:7.0f}bp "
          f"{mc_pik.effective_spread_bp - mc_cash.effective_spread_bp:+7.0f}")

print()
print("  lam Cash / lam PIK = flat hazard rates reproducing the MC model prices")
print("  d lam = PIK hazard premium (extra hazard the structural model")
print("          implicitly assigns to PIK over cash-pay)")

HAZARD-RATE PRICES: Cash-Pay vs PIK at Each Issuer's Flat lambda
  Issuer                     lam(bp)    HR Cash     HR PIK   d Price    MC Cash     MC PIK   d Price
  ------------------------------------------------------------------------------------------
  BB+ (Solid HY)                 150     113.13     115.94     +2.80     113.53     113.81     +0.29
  BB- (Mid HY)                   350     107.07     108.76     +1.69      99.02      99.02     +0.00
  B  (Weak HY)                   600      99.44      99.74     +0.30      78.97      76.37     -2.60
  B- (Stressed)                  900      90.50      89.21     -1.29      63.67      56.59     -7.08
  CCC (Deeply Stressed)         1400      77.41      73.97     -3.43      54.27      41.95    -12.32

  d Price = PIK minus Cash (negative = PIK trades cheaper)
  HR captures timing + notional; MC adds endogenous feedback.

IMPLIED HAZARD RATES: Flat lambda Backing Out Each Merton MC Price
  Issuer                     Base lam  lam Cas

## 8. Sensitivity: PIK Premium vs Asset Coverage

Sweep asset values from 110 to 220 (coverage 1.10x to 2.20x) and plot the PIK premium.

In [11]:
# Sweep coverage ratios
coverages = []
cash_spreads = []
pik_spreads = []
pik_premiums = []

for V in range(110, 225, 5):
    p = {"V": V, "vol": 0.30, "B": 100, "h0": 0.06, "R0": 0.35}
    m = make_merton(p)
    endo = EndogenousHazardSpec.power_law(p["h0"], p["B"] / p["V"], 2.0)
    drec = DynamicRecoverySpec.floored_inverse(p["R0"], NOTIONAL, 0.10)

    kw = dict(num_paths=15_000, seed=SEED, antithetic=True, time_steps_per_year=12)
    cash_cfg = MertonMcConfig(merton=m, **kw)
    pik_cfg  = MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec, **kw)

    rc = bond.price_merton_mc(config=cash_cfg, discount_rate=RISK_FREE, as_of=AS_OF)
    rp = bond.price_merton_mc(config=pik_cfg,  discount_rate=RISK_FREE, as_of=AS_OF)

    coverages.append(V / 100)
    cash_spreads.append(rc.effective_spread_bp)
    pik_spreads.append(rp.effective_spread_bp)
    pik_premiums.append(rp.effective_spread_bp - rc.effective_spread_bp)

print(f"{'Coverage':>10s}  {'Cash bp':>8s}  {'PIK bp':>8s}  {'Premium':>8s}")
print("-" * 40)
for i in range(0, len(coverages), 3):  # print every 3rd for readability
    print(f"{coverages[i]:>10.2f}x {cash_spreads[i]:>8.0f}  {pik_spreads[i]:>8.0f}  {pik_premiums[i]:>+8.0f}")

  Coverage   Cash bp    PIK bp   Premium
----------------------------------------
      1.10x      716       864      +149
      1.25x      296       392       +96
      1.40x       22        89       +67
      1.55x     -160      -110       +50
      1.70x     -296      -259       +38
      1.85x     -395      -366       +29
      2.00x     -465      -442       +24
      2.15x     -520      -501       +19


## 9. Toggle Strategy Comparison

Compare three toggle strategies for the B- issuer:
- **Threshold (hazard > 10%)** — Simple rule-based toggle
- **Stochastic (sigmoid)** — Smooth probability function of hazard rate
- **Optimal Exercise (nested MC)** — Forward-looking equity-maximising decision

In [12]:
p = PROFILES[3]  # B- (Stressed)
m = make_merton(p)
endo = EndogenousHazardSpec.power_law(p["h0"], p["B"] / p["V"], 2.0)
drec = DynamicRecoverySpec.floored_inverse(p["R0"], NOTIONAL, 0.10)
kw = dict(num_paths=NUM_PATHS, seed=SEED, antithetic=True, time_steps_per_year=12)

toggles = {
    "No Toggle (Cash)":       None,
    "No Toggle (Full PIK)":   None,  # special case
    "Threshold (h > 10%)": ToggleExerciseModel.threshold("hazard_rate", 0.10, "above"),
    "Stochastic (sigmoid)":   ToggleExerciseModel.stochastic("hazard_rate", -3.0, 25.0),
    "Optimal (nested MC)":    ToggleExerciseModel.optimal_exercise(
        nested_paths=200, equity_discount_rate=COUPON / 2, asset_vol=p["vol"], risk_free_rate=RISK_FREE,
    ),
}

print(f"B- Issuer: V={p['V']}, vol={p['vol']:.0%}, coverage={p['V']/p['B']:.2f}x")
print()
print(f"  {'Strategy':<25s}  {'Price':>7s}  {'Spread':>7s}  {'E[Loss]':>7s}  "
      f"{'DefRate':>7s}  {'PIK%':>6s}  {'TermNtl':>8s}")
print(f"  {'-' * 72}")

for label, toggle in toggles.items():
    if label == "No Toggle (Full PIK)":
        # Full PIK: use is_pik=True via endogenous + dynamic (no toggle needed, PIK fraction = 100%)
        cfg = MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec, **kw)
    elif toggle is None:
        # Cash-pay: no endogenous/dynamic
        cfg = MertonMcConfig(merton=m, **kw)
    else:
        cfg = MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec,
                             toggle_model=toggle, **kw)

    r = bond.price_merton_mc(config=cfg, discount_rate=RISK_FREE, as_of=AS_OF)
    print(f"  {label:<25s}  {r.clean_price_pct:7.2f}  {r.effective_spread_bp:6.0f}bp  "
          f"{r.expected_loss:7.2%}  {r.default_rate:7.2%}  {r.average_pik_fraction:5.1%}  "
          f"{r.avg_terminal_notional:8.1f}")

B- Issuer: V=125, vol=35%, coverage=1.25x

  Strategy                     Price   Spread  E[Loss]  DefRate    PIK%   TermNtl
  ------------------------------------------------------------------------
  No Toggle (Cash)             63.67     453bp   45.81%   74.78%   0.0%     100.0
  No Toggle (Full PIK)         56.59     689bp   51.83%   74.78%   0.0%     100.0
  Threshold (h > 10%)          55.08     743bp   53.11%   74.78%  13.0%     102.1
  Stochastic (sigmoid)         55.08     743bp   53.12%   74.78%  20.7%     106.8
  Optimal (nested MC)          55.48     728bp   52.77%   74.78%  17.2%     106.2


## 10. Sensitivity: Asset Volatility Impact

For a fixed leverage (1.40x coverage), sweep asset vol from 15% to 45% to see how vol amplifies PIK risk.

In [13]:
print(f"{'Vol':>5s}  {'Cash Price':>10s}  {'PIK Price':>10s}  {'Cash Sprd':>9s}  {'PIK Sprd':>9s}  {'Premium':>8s}")
print("-" * 60)

for vol_pct in range(15, 50, 5):
    vol = vol_pct / 100
    p = {"V": 140, "vol": vol, "B": 100, "h0": 0.06, "R0": 0.35}
    m = make_merton(p)
    endo = EndogenousHazardSpec.power_law(p["h0"], p["B"] / p["V"], 2.0)
    drec = DynamicRecoverySpec.floored_inverse(p["R0"], NOTIONAL, 0.10)

    kw = dict(num_paths=15_000, seed=SEED, antithetic=True, time_steps_per_year=12)
    rc = bond.price_merton_mc(config=MertonMcConfig(merton=m, **kw),
                              discount_rate=RISK_FREE, as_of=AS_OF)
    rp = bond.price_merton_mc(
        config=MertonMcConfig(merton=m, endogenous_hazard=endo, dynamic_recovery=drec, **kw),
        discount_rate=RISK_FREE, as_of=AS_OF,
    )

    prem = rp.effective_spread_bp - rc.effective_spread_bp
    print(f"{vol:>5.0%}  {rc.clean_price_pct:>10.2f}  {rp.clean_price_pct:>10.2f}  "
          f"{rc.effective_spread_bp:>8.0f}bp {rp.effective_spread_bp:>8.0f}bp {prem:>+8.0f}bp")

  Vol  Cash Price   PIK Price  Cash Sprd   PIK Sprd   Premium
------------------------------------------------------------
  15%      107.80      107.12      -600bp     -588bp      +13bp
  20%       96.84       95.42      -386bp     -356bp      +30bp
  25%       86.88       84.80      -169bp     -120bp      +49bp
  30%       78.96       76.37        22bp       89bp      +67bp
  35%       72.53       69.53       192bp      277bp      +85bp
  40%       67.72       64.41       330bp      430bp     +100bp
  45%       63.65       60.09       454bp      569bp     +115bp


## Conclusions

**Key findings:**

1. **PIK premium is non-linear in credit quality.** For healthy issuers (BB+, 2x coverage), PIK barely matters — defaults are rare, so PIK accrual doesn’t spiral.  But for stressed issuers (CCC, 1.15x coverage), the PIK premium can exceed 500bp.

2. **The feedback loop drives PIK risk.** The combination of endogenous hazard (leverage raises default intensity) and dynamic recovery (higher notional dilutes recovery) creates a self-reinforcing spiral that dramatically worsens PIK bond economics for weak credits.

3. **Simple hazard rates miss the story.** A flat hazard-rate model captures the timing and notional effects of PIK but grossly underestimates the premium for stressed issuers.  For CCC credits, the implied hazard rate for PIK is ~700bp above the cash implied hazard — almost entirely driven by the structural feedback that simple models omit.

4. **Toggle is not free.** Even though the toggle preserves the borrower’s *option* to pay cash, the investor still bears negative convexity: the borrower PIKs precisely when credit deteriorates.  The toggle spread premium sits between cash and full PIK.

5. **Volatility amplifies PIK risk.** Higher asset vol increases both the frequency and severity of the leverage spiral, making PIK premiums disproportionately larger.

6. **The original question** (breakeven for a B-rated, 600bp spread PIK deal): the MC engine shows a PIK premium of ~67–290bp above cash, depending on exact credit quality, implying a total spread of ~90–740bp.  A 600bp deal would be appropriately priced for a **B– to CCC** issuer.